# Evaluación de Modelos de Lenguaje
### Métricas, Análisis de Errores y Reporte
**Proyecto I: Introducción a LLMs — Facultad de Ciencias, UNAM — Semestre 2026-2**

---

> **Dónde estamos:** Tenemos modelos funcionando — un clasificador de sentimiento con LoRA y un chatbot con RAG. Ahora necesitamos medir qué tan bien funcionan de forma rigurosa. Sin métricas claras, no hay ciencia — solo intuición.

**Mapa de la sesión:**
```
Parte 1: Métricas para clasificación — proyecto de sentimiento   (20 min)
Parte 2: Métricas para generación — proyecto de chatbot          (20 min)
Parte 3: Análisis de errores                                     (10 min)
Parte 4: Cómo reportar resultados en el reporte de titulación    (10 min)
```

**Sin GPU necesaria** — esta sesión es principalmente análisis.

In [ ]:
!pip install scikit-learn rouge-score bert-score matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_curve, auc, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

print("Listo.")

---
## Parte 1: Métricas para Clasificación
### *(Proyecto: análisis de sentimiento en reseñas Google Maps)*

### ¿Por qué no solo accuracy?

Imagina un dataset con 90% de reseñas positivas. Un modelo que predice *siempre positivo* tendría 90% de accuracy — pero sería inútil. Para clasificación con clases desbalanceadas necesitamos métricas más informativas.

Las cuatro métricas fundamentales se derivan de la **matriz de confusión**:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

Para multiclase se promedian de tres formas:
- **macro:** promedio simple entre clases (trata todas igual — penaliza clases minoritarias ignoradas)
- **weighted:** promedio ponderado por frecuencia (favorece clases mayoritarias)
- **micro:** agrega TP, FP, FN antes de calcular (equivale a accuracy en multiclase)

In [ ]:
# Intuición: el mismo accuracy puede esconder resultados muy distintos

np.random.seed(42)
CLASES = ['negativo', 'neutro', 'positivo']

# Escenario A: modelo bien balanceado
y_real = np.array([0]*30 + [1]*30 + [2]*40)
y_pred_A = np.array(
    [0]*25 + [1]*3  + [2]*2  +   # negativo: 25 correctos
    [0]*4  + [1]*23 + [2]*3  +   # neutro: 23 correctos
    [0]*2  + [1]*3  + [2]*35     # positivo: 35 correctos
)

# Escenario B: modelo que ignora la clase minoritaria
y_pred_B = np.array(
    [2]*30 +   # predice todo 'positivo' para negativos
    [2]*30 +   # predice todo 'positivo' para neutros
    [2]*40     # correctos para positivos
)

acc_A = (y_pred_A == y_real).mean()
acc_B = (y_pred_B == y_real).mean()
f1_A  = f1_score(y_real, y_pred_A, average='macro')
f1_B  = f1_score(y_real, y_pred_B, average='macro')

print("Comparación de dos modelos con accuracy similar:")
print(f"{'Métrica':>20} | {'Modelo A (balanceado)':>22} | {'Modelo B (sesgado)':>20}")
print("-" * 68)
print(f"{'Accuracy':>20} | {acc_A:>22.3f} | {acc_B:>20.3f}")
print(f"{'F1-macro':>20} | {f1_A:>22.3f} | {f1_B:>20.3f}")
print()
print("Modelo B: accuracy alta pero F1-macro bajo.")
print("→ Ignora completamente las clases 'negativo' y 'neutro'.")
print("→ Para Google Maps, esto significa que nunca detecta reseñas malas.")
print("→ F1-macro penaliza este comportamiento; accuracy no.")

In [ ]:
# Datos simulados: baseline vs LoRA fine-tuning
# Sustituir con los resultados reales del notebook de LoRA

np.random.seed(7)
n_test = 262  # tamaño del test set del notebook de LoRA

# Distribución real: 25% neg, 20% neu, 55% pos (típico en Google Maps)
y_test = np.concatenate([
    np.zeros(66, dtype=int),
    np.ones(52, dtype=int),
    np.full(144, 2, dtype=int)
])
np.random.shuffle(y_test)

# Baseline: modelo preentrenado sin fine-tuning (peor en neutro)
def simular_predicciones(y_true, acc_neg, acc_neu, acc_pos):
    preds = np.zeros_like(y_true)
    for i, label in enumerate(y_true):
        acc = [acc_neg, acc_neu, acc_pos][label]
        if np.random.random() < acc:
            preds[i] = label
        else:
            opciones = [c for c in range(3) if c != label]
            preds[i] = np.random.choice(opciones)
    return preds

y_baseline = simular_predicciones(y_test, acc_neg=0.65, acc_neu=0.45, acc_pos=0.78)
y_lora     = simular_predicciones(y_test, acc_neg=0.82, acc_neu=0.70, acc_pos=0.89)

# Comparar
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusión — Baseline vs LoRA', fontsize=13, fontweight='bold')

for ax, y_pred, titulo in zip(axes,
                               [y_baseline, y_lora],
                               ['Baseline (sin fine-tuning)', 'LoRA Fine-tuning']):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASES, fontsize=11)
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASES, fontsize=11)
    ax.set_xlabel('Predicción', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)

    f1 = f1_score(y_test, y_pred, average='macro')
    ax.set_title(f'{titulo}\nF1-macro = {f1:.3f}', fontsize=11, fontweight='bold')

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.0%})',
                    ha='center', va='center', fontsize=10,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, label='Proporción')

plt.tight_layout()
plt.show()

print("Leer la matriz: fila = clase real, columna = predicción.")
print("La diagonal principal son los aciertos.")
print("Las celdas fuera de la diagonal son los errores — ¿cuáles son más graves?")

In [ ]:
# Reporte completo por clase

print("BASELINE — Reporte de clasificación:")
print(classification_report(y_test, y_baseline, target_names=CLASES))

print("\nLoRA FINE-TUNING — Reporte de clasificación:")
print(classification_report(y_test, y_lora, target_names=CLASES))

# Tabla comparativa compacta
print("\nResumen comparativo:")
print(f"{'Métrica':>25} | {'Baseline':>10} | {'LoRA':>10} | {'Δ':>8}")
print("-" * 60)

metricas = [
    ('Accuracy',       (y_baseline == y_test).mean(), (y_lora == y_test).mean()),
    ('F1-macro',       f1_score(y_test, y_baseline, average='macro'),
                       f1_score(y_test, y_lora, average='macro')),
    ('F1-weighted',    f1_score(y_test, y_baseline, average='weighted'),
                       f1_score(y_test, y_lora, average='weighted')),
    ('F1 negativo',    f1_score(y_test, y_baseline, average=None)[0],
                       f1_score(y_test, y_lora, average=None)[0]),
    ('F1 neutro',      f1_score(y_test, y_baseline, average=None)[1],
                       f1_score(y_test, y_lora, average=None)[1]),
    ('F1 positivo',    f1_score(y_test, y_baseline, average=None)[2],
                       f1_score(y_test, y_lora, average=None)[2]),
]

for nombre, val_base, val_lora in metricas:
    delta = val_lora - val_base
    signo = '+' if delta >= 0 else ''
    print(f"{nombre:>25} | {val_base:>10.3f} | {val_lora:>10.3f} | {signo}{delta:>7.3f}")

In [ ]:
# Curvas Precision-Recall por clase
# Más informativas que ROC cuando el dataset está desbalanceado

from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression

# Simular probabilidades para las curvas
np.random.seed(42)

def simular_probs(y_true, y_pred, suavizado=0.15):
    """Genera probabilidades simuladas consistentes con las predicciones."""
    n, k = len(y_true), 3
    probs = np.random.dirichlet(np.ones(k) * suavizado, size=n)
    for i, (yt, yp) in enumerate(zip(y_true, y_pred)):
        probs[i, yp] += 0.6
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs

probs_baseline = simular_probs(y_test, y_baseline)
probs_lora     = simular_probs(y_test, y_lora, suavizado=0.08)
y_bin          = label_binarize(y_test, classes=[0, 1, 2])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Curvas Precision-Recall por clase', fontsize=12, fontweight='bold')

for j, clase in enumerate(CLASES):
    for probs, label, color in [
        (probs_baseline, 'Baseline', '#90CAF9'),
        (probs_lora,     'LoRA',     '#1565C0')
    ]:
        prec, rec, _ = precision_recall_curve(y_bin[:, j], probs[:, j])
        ap = auc(rec, prec)
        axes[j].plot(rec, prec, color=color, linewidth=2, label=f'{label} (AP={ap:.2f})')

    baseline_prec = precision_score(y_test, y_baseline, average=None, zero_division=0)[j]
    axes[j].set_title(f'Clase: {clase}', fontsize=11, fontweight='bold')
    axes[j].set_xlabel('Recall')
    axes[j].set_ylabel('Precision')
    axes[j].legend(fontsize=9)
    axes[j].grid(True, alpha=0.3)
    axes[j].set_xlim(0, 1); axes[j].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("Área bajo la curva PR (AP): mayor = mejor.")
print("Para clase 'neutro' el AP suele ser más bajo — es la más difícil de clasificar.")
print("Esto es normal y debe explicarse en el reporte.")

---
## Parte 2: Métricas para Generación
### *(Proyecto: chatbot de joyería)*

Evaluar texto generado es fundamentalmente más difícil que clasificación: no hay una sola respuesta correcta. Las métricas automáticas son aproximaciones — siempre deben complementarse con evaluación humana.

### Las tres métricas principales

| Métrica | Mide | Limitación |
|:---|:---|:---|
| **ROUGE** | Solapamiento de n-gramas con referencia | No entiende paráfrasis |
| **BERTScore** | Similitud semántica con embeddings | Costoso, menos interpretable |
| **Perplejidad** | Qué tan probable es el texto para el modelo | No mide calidad real |

Para chatbots de dominio cerrado (catálogo de productos), lo más importante es verificar **factual correctness** — que los datos del catálogo (precios, disponibilidad) se reproduzcan correctamente.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

# Pares referencia / generación para el chatbot de joyería
# La referencia es la respuesta ideal (escrita por el humano)
# La generación es lo que produjo el modelo

pares_evaluacion = [
    {
        'pregunta':    '¿Cuánto cuestan los aretes de plata con turquesa?',
        'referencia':  'Los aretes de plata 925 con turquesa natural cuestan $650 MXN y están disponibles en azul y verde.',
        'generacion':  'Tenemos aretes de plata 925 con turquesa natural a $650 MXN, disponibles en color azul y verde.',
        'categoria':   'precio correcto'
    },
    {
        'pregunta':    '¿Hacen envíos a Monterrey?',
        'referencia':  'Sí, enviamos a toda la República Mexicana. El costo es $120 MXN y el tiempo de entrega es de 5-7 días hábiles.',
        'generacion':  'Claro que sí, hacemos envíos a todo el país incluyendo Monterrey. El envío cuesta $120 pesos y tarda entre 5 y 7 días hábiles en llegar.',
        'categoria':   'logística correcta'
    },
    {
        'pregunta':    '¿Tienen anillos de oro?',
        'referencia':  'Lo siento, no trabajamos con oro macizo. Nuestros materiales son plata 925 y cobre artesanal.',
        'generacion':  'Contamos con hermosos anillos de oro de 18 kilates desde $2,500 MXN.',  # alucinación
        'categoria':   'alucinación'
    },
    {
        'pregunta':    '¿Puedo pedir una pulsera personalizada?',
        'referencia':  'Sí, hacemos pedidos personalizados con tiempo adicional de 7-10 días y 50% de anticipo.',
        'generacion':  'Por supuesto, realizamos piezas personalizadas. Se requiere un anticipo del 50% y el tiempo adicional es de 7 a 10 días hábiles.',
        'categoria':   'personalización correcta'
    },
]

print("ROUGE scores — Chatbot de Joyería")
print(f"{'Pregunta':>45} | {'R-1':>6} | {'R-2':>6} | {'R-L':>6} | Categoría")
print("-" * 90)

todos_r1, todos_r2, todos_rl = [], [], []
for par in pares_evaluacion:
    scores = scorer.score(par['referencia'], par['generacion'])
    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rl = scores['rougeL'].fmeasure
    todos_r1.append(r1); todos_r2.append(r2); todos_rl.append(rl)
    print(f"{par['pregunta'][:45]:>45} | {r1:>6.3f} | {r2:>6.3f} | {rl:>6.3f} | {par['categoria']}")

print("-" * 90)
print(f"{'PROMEDIO':>45} | {np.mean(todos_r1):>6.3f} | {np.mean(todos_r2):>6.3f} | {np.mean(todos_rl):>6.3f}")
print()
print("Observar: la 'alucinación' puede tener ROUGE alto si las palabras coinciden")
print("pero el precio o producto son inventados. ROUGE no detecta errores factuales.")
print("→ Por eso la evaluación humana con rúbrica es indispensable.")

In [ ]:
# BERTScore: similitud semántica usando embeddings
# Detecta paráfrasis que ROUGE pierde

from bert_score import score as bert_score

referencias  = [p['referencia'] for p in pares_evaluacion]
generaciones = [p['generacion'] for p in pares_evaluacion]
categorias   = [p['categoria']  for p in pares_evaluacion]

print("Calculando BERTScore (primera vez descarga el modelo)...")
P, R, F1 = bert_score(
    generaciones, referencias,
    lang='es',
    verbose=False
)

print("\nBERTScore — Chatbot de Joyería")
print(f"{'Pregunta':>45} | {'P':>6} | {'R':>6} | {'F1':>6} | Categoría")
print("-" * 90)

for i, par in enumerate(pares_evaluacion):
    print(f"{par['pregunta'][:45]:>45} | {P[i].item():>6.3f} | {R[i].item():>6.3f} | {F1[i].item():>6.3f} | {par['categoria']}")

print("-" * 90)
print(f"{'PROMEDIO':>45} | {P.mean().item():>6.3f} | {R.mean().item():>6.3f} | {F1.mean().item():>6.3f}")
print()
print("BERTScore típicamente da valores altos (>0.85) incluso para paráfrasis.")
print("Valores < 0.80 suelen indicar respuestas semánticamente incorrectas.")

In [ ]:
# Métricas de dominio específico — más relevantes que ROUGE/BERTScore
# para un chatbot de e-commerce

import re

PRECIOS_CATALOGO = {320, 420, 480, 550, 580, 650, 750, 890, 980, 1100, 1200}

def extraer_precios(texto):
    patrones = re.findall(r'\$?([\d,]+)(?:\s*(?:MXN|pesos))?', texto)
    precios = set()
    for p in patrones:
        try:
            v = int(p.replace(',', ''))
            if 100 <= v <= 5000:
                precios.add(v)
        except:
            pass
    return precios

def evaluar_factual(generacion, referencia):
    """Evalúa si los datos del catálogo se reprodujeron correctamente."""
    precios_gen = extraer_precios(generacion)
    precios_ref = extraer_precios(referencia)

    alucinados = precios_gen - PRECIOS_CATALOGO
    correctos  = precios_gen & PRECIOS_CATALOGO
    omitidos   = precios_ref - precios_gen

    return {
        'precios_correctos':  correctos,
        'precios_alucinados': alucinados,
        'precios_omitidos':   omitidos,
        'factual_ok': len(alucinados) == 0
    }

print("Evaluación factual — Chatbot de Joyería")
print("=" * 60)
n_ok = 0
for par in pares_evaluacion:
    resultado = evaluar_factual(par['generacion'], par['referencia'])
    estado = "✓ OK" if resultado['factual_ok'] else "✗ ALUCINACIÓN"
    print(f"\nPregunta: {par['pregunta'][:55]}")
    print(f"  Estado: {estado}")
    if resultado['precios_correctos']:
        print(f"  Precios correctos:  {resultado['precios_correctos']}")
    if resultado['precios_alucinados']:
        print(f"  Precios inventados: {resultado['precios_alucinados']}  ← ERROR GRAVE")
    if resultado['precios_omitidos']:
        print(f"  Precios omitidos:   {resultado['precios_omitidos']}")
    if resultado['factual_ok']:
        n_ok += 1

print(f"\nPrecisión factual: {n_ok}/{len(pares_evaluacion)} ({n_ok/len(pares_evaluacion)*100:.0f}%)")
print()
print("Para un chatbot de e-commerce, la precisión factual es la métrica más importante.")
print("Un precio incorrecto puede generar conflictos reales con clientes.")

In [ ]:
# Perplejidad: ¿qué mide exactamente?
# Útil para comparar versiones del mismo modelo, no modelos distintos

print("Perplejidad — interpretación intuitiva")
print("=" * 55)
print()
print("Perplejidad = exp(cross-entropy promedio por token)")
print()
print("Interpretación: si PPL = k, el modelo está tan confundido")
print("como si tuviera que elegir entre k opciones equiprobables")
print("en cada paso.")
print()

ejemplos_ppl = [
    (2,    "Modelo casi perfecto — cada token es casi determinista"),
    (10,   "Modelo muy bueno — pocas opciones por token"),
    (50,   "Modelo razonable — rango típico de LMs bien entrenados"),
    (200,  "Modelo débil — mucha incertidumbre por token"),
    (1000, "Modelo malo o dominio incorrecto"),
]

print(f"{'PPL':>8} | Interpretación")
print("-" * 65)
for ppl, desc in ejemplos_ppl:
    print(f"{ppl:>8} | {desc}")

print()
print("Cuándo usar perplejidad en tu proyecto:")
print("  ✓ Comparar dos checkpoints del mismo modelo durante entrenamiento")
print("  ✓ Verificar que el modelo no está sobre-ajustando (train PPL << val PPL)")
print("  ✗ Comparar modelos distintos entre sí")
print("  ✗ Medir calidad real de respuestas de chatbot")

---
## Parte 3: Análisis de Errores

El análisis de errores es lo más valioso para mejorar un modelo. La pregunta no es solo *cuántos* errores comete, sino *qué tipo* de errores y *por qué*.

In [ ]:
# Análisis de errores para clasificación de sentimiento

# Dataset de ejemplo con texto real para analizar
resenas_test_ejemplo = [
    ("Excelente lugar, todo perfecto.",                     2, 2, "claro"),
    ("Horrible, nunca regreso.",                            0, 0, "claro"),
    ("Estuvo bien aunque tardaron mucho.",                  1, 2, "ambiguo — aspectos mezclados"),
    ("No es tan malo como dicen.",                          1, 0, "negación — difícil para el modelo"),
    ("Cumple su función, nada más.",                        1, 2, "neutro confundido con positivo"),
    ("La comida rica pero el servicio fatal.",              1, 0, "aspectos contradictorios"),
    ("¿Por qué tiene 5 estrellas? No entiendo.",            0, 1, "ironía — muy difícil"),
    ("Bueno para lo que es, sin esperar más.",              1, 2, "matizado"),
]

TIPOS_ERROR = {
    "claro":                      "#2E7D32",
    "ambiguo — aspectos mezclados": "#F9A825",
    "negación — difícil para el modelo": "#E65100",
    "neutro confundido con positivo": "#F9A825",
    "aspectos contradictorios":    "#E65100",
    "ironía — muy difícil":        "#C62828",
    "matizado":                    "#F9A825",
}

print("Taxonomía de errores — Clasificador de Sentimiento")
print("=" * 70)
print(f"{'Reseña':>45} | {'Real':>8} | {'Pred':>8} | {'Tipo'}")
print("-" * 85)

errores_por_tipo = {}
for resena, real, pred, tipo in resenas_test_ejemplo:
    ok = "✓" if real == pred else "✗"
    real_str = CLASES[real]
    pred_str = CLASES[pred]
    print(f"{resena[:45]:>45} | {real_str:>8} | {pred_str:>8} {ok} | {tipo}")
    if real != pred:
        errores_por_tipo[tipo] = errores_por_tipo.get(tipo, 0) + 1

if errores_por_tipo:
    print(f"\nErrores por tipo:")
    for tipo, n in sorted(errores_por_tipo.items(), key=lambda x: -x[1]):
        print(f"  {tipo}: {n}")

print()
print("Los errores en negación e ironía son los más difíciles de resolver")
print("porque requieren comprensión pragmática profunda del lenguaje.")
print("Documentar estos casos en el reporte demuestra análisis crítico.")

In [ ]:
# Taxonomía de errores para el chatbot

errores_chatbot = [
    {
        'tipo': 'Alucinación factual',
        'descripcion': 'El modelo inventa información que no está en el catálogo',
        'ejemplo': 'Usuario: ¿tienen anillos? | Modelo: Sí, tenemos anillos de oro desde $2,500',
        'causa': 'El retriever no encontró contexto relevante y el modelo "completó" con conocimiento general',
        'solución': 'Mejorar el umbral de similitud; agregar instrucción explícita de no inventar',
        'severidad': 'ALTA — puede generar conflictos con clientes'
    },
    {
        'tipo': 'Precio incorrecto',
        'descripcion': 'El modelo menciona un precio distinto al del catálogo',
        'ejemplo': 'Usuario: ¿cuánto cuestan los aretes de luna? | Modelo: $280 MXN [real: $320]',
        'causa': 'El chunk recuperado tiene varios precios y el modelo selecciona el incorrecto',
        'solución': 'Fragmentar el catálogo en chunks más pequeños (1 producto por chunk)',
        'severidad': 'ALTA — información incorrecta'
    },
    {
        'tipo': 'Respuesta fuera de idioma',
        'descripcion': 'El modelo responde en inglés o mezcla idiomas',
        'ejemplo': 'Usuario: ¿tienen envíos gratis? | Modelo: Yes, we offer free shipping over $1,500',
        'causa': 'El system prompt no refuerza suficientemente el idioma',
        'solución': 'Reforzar en el prompt: "SIEMPRE responde en español, sin excepción"',
        'severidad': 'MEDIA — afecta experiencia de usuario'
    },
    {
        'tipo': 'Omisión de información relevante',
        'descripcion': 'El modelo responde correctamente pero omite datos importantes',
        'ejemplo': 'Usuario: ¿hacen envíos? | Modelo: Sí [omite costo y tiempo]',
        'causa': 'k=3 chunks puede no incluir toda la información necesaria',
        'solución': 'Aumentar k o ajustar el prompt para pedir respuestas más completas',
        'severidad': 'BAJA — información incompleta pero no incorrecta'
    },
]

print("Taxonomía de errores — Chatbot de Joyería")
print("=" * 65)
for i, error in enumerate(errores_chatbot, 1):
    print(f"\n{i}. {error['tipo']} [{error['severidad']}]")
    print(f"   Descripción: {error['descripcion']}")
    print(f"   Ejemplo:     {error['ejemplo']}")
    print(f"   Causa:       {error['causa']}")
    print(f"   Solución:    {error['solución']}")

print()
print("Esta taxonomía va en la sección 'Análisis de Errores' del reporte.")

---
## Parte 4: Cómo Reportar Resultados

El reporte técnico con formato de tesis requiere presentar los resultados de forma clara y reproducible. A continuación, la estructura recomendada para la sección de experimentos.

In [ ]:
# Tabla de resultados lista para copiar al reporte
# Formato estándar en papers de NLP

print("TABLA DE RESULTADOS — para incluir en el reporte")
print("=" * 70)
print()
print("Tabla 1: Resultados de clasificación de sentimiento en test set")
print()
print(f"{'Modelo':<35} | {'Acc':>5} | {'F1-M':>5} | {'F1-W':>5} | {'F1-neg':>6} | {'F1-neu':>6} | {'F1-pos':>6}")
print("-" * 80)

modelos_tabla = [
    ("Baseline (nlptown/bert-multilingual)",
     y_test, y_baseline),
    ("LoRA r=8, lr=2e-4, epochs=3",
     y_test, y_lora),
]

for nombre, y_true, y_pred in modelos_tabla:
    acc    = (y_pred == y_true).mean()
    f1m    = f1_score(y_true, y_pred, average='macro')
    f1w    = f1_score(y_true, y_pred, average='weighted')
    f1_cls = f1_score(y_true, y_pred, average=None)
    print(f"{nombre:<35} | {acc:>5.3f} | {f1m:>5.3f} | {f1w:>5.3f} | {f1_cls[0]:>6.3f} | {f1_cls[1]:>6.3f} | {f1_cls[2]:>6.3f}")

print()
print("Notas al pie de la tabla (obligatorias en el reporte):")
print("  - Dataset: [nombre y tamaño real del dataset de Google Maps]")
print("  - Métrica principal: F1-macro (justificación: clases desbalanceadas)")
print("  - LoRA target_modules: query, value | r=8 | alpha=16")
print("  - Resultados promediados sobre [N] corridas con semillas 42, 0, 7")
print()
print("IMPORTANTE: Siempre reportar métricas en el TEST set, nunca en validación.")
print("El val set se usa solo para seleccionar el mejor checkpoint.")

In [ ]:
# Figura de resultados para el reporte — lista para exportar

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Resultados del Proyecto — Análisis de Sentimiento en Reseñas',
             fontsize=13, fontweight='bold')

# ── F1 por clase ──
x = np.arange(3)
width = 0.35
f1_base = f1_score(y_test, y_baseline, average=None)
f1_lora = f1_score(y_test, y_lora, average=None)

axes[0].bar(x - width/2, f1_base, width, label='Baseline', color='#90CAF9', edgecolor='#333')
axes[0].bar(x + width/2, f1_lora, width, label='LoRA',     color='#1565C0', edgecolor='#333')
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASES, fontsize=11)
axes[0].set_ylabel('F1-score'); axes[0].set_ylim(0, 1)
axes[0].set_title('F1 por clase', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# ── Métricas globales ──
nombres_met = ['Accuracy', 'F1-macro', 'F1-weighted']
vals_base = [
    (y_baseline == y_test).mean(),
    f1_score(y_test, y_baseline, average='macro'),
    f1_score(y_test, y_baseline, average='weighted')
]
vals_lora = [
    (y_lora == y_test).mean(),
    f1_score(y_test, y_lora, average='macro'),
    f1_score(y_test, y_lora, average='weighted')
]
x2 = np.arange(3)
axes[1].bar(x2 - width/2, vals_base, width, label='Baseline', color='#90CAF9', edgecolor='#333')
axes[1].bar(x2 + width/2, vals_lora, width, label='LoRA',     color='#1565C0', edgecolor='#333')
axes[1].set_xticks(x2); axes[1].set_xticklabels(nombres_met, fontsize=10)
axes[1].set_ylim(0, 1); axes[1].set_title('Métricas globales', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# ── Mejora relativa ──
mejoras = [(v_l - v_b) / v_b * 100 for v_b, v_l in zip(vals_base, vals_lora)]
colores_barra = ['#2E7D32' if m >= 0 else '#C62828' for m in mejoras]
axes[2].bar(nombres_met, mejoras, color=colores_barra, edgecolor='#333')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Mejora relativa LoRA vs Baseline (%)', fontweight='bold')
axes[2].set_ylabel('Mejora (%)')
for i, m in enumerate(mejoras):
    axes[2].text(i, m + (0.3 if m >= 0 else -0.8),
                 f'{m:+.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('resultados_sentimiento.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada como 'resultados_sentimiento.png'")
print("Incluir en el reporte con pie de figura descriptivo.")

In [ ]:
# Checklist del reporte técnico

checklist = {
    "Sección de Experimentos": [
        "Descripción del dataset (tamaño, fuente, distribución de clases)",
        "Descripción del baseline y justificación de la elección del modelo",
        "Hiperparámetros del fine-tuning (r, alpha, lr, epochs, batch size)",
        "Métrica principal y justificación (¿por qué F1-macro y no accuracy?)",
        "Tabla de resultados en test set (no en validación)",
        "Gráficas: curvas de aprendizaje, matriz de confusión, F1 por clase",
    ],
    "Sección de Análisis": [
        "Análisis de errores con ejemplos concretos",
        "Taxonomía de errores (tipos y causas)",
        "Comparación cualitativa: ¿qué mejoró y qué sigue fallando?",
        "Discusión de limitaciones del modelo",
    ],
    "Reproducibilidad": [
        "Semilla aleatoria fija en el código",
        "Tarjeta de experimento guardada en JSON",
        "Código disponible en repositorio de GitHub",
        "Instrucciones para reproducir los resultados",
    ]
}

print("CHECKLIST — Reporte Técnico con Formato de Tesis")
print("=" * 55)
for seccion, items in checklist.items():
    print(f"\n{seccion}:")
    for item in items:
        print(f"  ☐ {item}")

---
## Resumen

| Proyecto | Métrica principal | Métricas complementarias |
|:---|:---|:---|
| Sentimiento Google Maps | F1-macro | F1 por clase, Precision-Recall, matriz de confusión |
| Chatbot joyería | Precisión factual + rúbrica | ROUGE-L, BERTScore-F1 |
| Asistente contable (RAG) | Precisión de recuperación | F1 de respuesta, cobertura de fuentes |

**Regla de oro:** reporta siempre la métrica en el **test set**, justifica tu elección de métrica principal, e incluye ejemplos cualitativos de errores. Los números solos no cuentan la historia completa.

**Próxima sesión:** Demo final y presentación de proyectos.